## Setup: imports, paths, study constants, shared helpers, phase colours

In [1]:
# Compact public exports; restore model context for existing figure code only.
def _read_result_csv(path, *args, **kwargs):
    result_table = pd.read_csv(path, *args, **kwargs)
    if "q_value" not in result_table.columns:
        return result_table
    from pathlib import Path
    aliases = {"q_value": "p_value_adjusted", "FDR_significant": "significant_adjusted",
               "predictor": "term", "comparison": "contrast", "grid_id": "hotspot_id",
               "n_grids": "n_hotspots", "test_statistic": "statistic"}
    result_table = result_table.rename(columns={key:current_values for key,current_values in aliases.items() if key in result_table.columns})
    source_path = Path(path)
    info = source_path.parent / (source_path.parent.name + "_run_info.csv")
    if info.exists():
        meta = pd.read_csv(info, dtype=str, keep_default_na=False)
        if set(["section", "item", "field", "value"]).issubset(meta.columns):
            prefix = source_path.name + "::"
            rows = meta[(meta.section == "result_context") & meta.item.str.startswith(prefix)]
            reference_fields = ["estimate", "std_error", "ci_lower", "ci_upper", "df", "statistic",
                                "p_value", "p_value_adjusted", "significant_adjusted", "n_observations", "n_participants"]
            promoted60 = source_path.name.endswith("Sensitivity_60s.csv") and rows.field.str.startswith("reference10s__").any()
            live60 = set()
            if promoted60:
                for field in reference_fields:
                    live60.add(field + "_60s")
                    result_table[field + "_60s"] = result_table[field] if field in result_table.columns else None
                    if field in result_table.columns:
                        result_table = result_table.drop(columns=[field])
            for field, group in rows.groupby("field", sort=False):
                if field in live60:
                    continue
                if field not in result_table.columns:
                    result_table[field] = None
                for item, value in zip(group.item, group.value):
                    index = int(item.rsplit("::", 1)[1]) - 1
                    if 0 <= index < len(result_table):
                        result_table.at[index, field] = value
            if promoted60:
                for field in reference_fields:
                    reference = "reference10s__" + field
                    result_table[field] = result_table[reference] if reference in result_table.columns else None
                    if reference in result_table.columns:
                        result_table = result_table.drop(columns=[reference])
    # Match pandas' original inference of numeric and Boolean fields.
    return pd.read_csv(io.StringIO(result_table.to_csv(index=False)), *args, **kwargs)

import io
import sys, os, re, hashlib, subprocess, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, Patch, Rectangle
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

from pathlib import Path

# ---- paths ----
# ROOT = project folder "Pranav Paper3_Final". The notebook lives in ROOT\scripts, so ROOT is the parent of the
# notebook's folder; the absolute PC path is the fallback.
_FALLBACK_ROOT = Path(r"C:\Users\sanga\Desktop\Pranav Paper3_Final")
_MERGED_REL = Path("output") / "merged_all_11participants_8min.csv"
ROOT = next((candidate_root for candidate_root in (Path.cwd().parent, Path.cwd(), _FALLBACK_ROOT) if (candidate_root / _MERGED_REL).exists()), _FALLBACK_ROOT)
OUT = ROOT / "Figures"
os.makedirs(OUT, exist_ok=True)
print("ROOT =", ROOT)
print("OUT  =", OUT)

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5, "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.5, "axes.linewidth": 0.6,
    "xtick.major.width": 0.5, "ytick.major.width": 0.5, "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "savefig.dpi": 300})

# ---- study constants (Paper3_common.R) ----
PARTICIPANTS = ["P4", "P8", "P9", "P10", "P11", "P12", "P13", "P14", "P15", "P16", "P17"]
PHASES = ["BikeU", "WalkU", "BikeG", "WalkG"]
PHASE_COL = {"BikeU": "#a50f15", "WalkU": "#f08a5d", "BikeG": "#006d2c", "WalkG": "#74c476"}
PHASE_LAB = {"BikeU": "Bike-Urban", "WalkU": "Walk-Urban", "BikeG": "Bike-Green", "WalkG": "Walk-Green"}
OUTCOMES = ["empatica_heart_rate", "empatica_eda_tonic", "empatica_eda_phasic", "empatica_skin_temperature",
    "empatica_hrv_td_sdnn", "empatica_hrv_td_rmssd", "empatica_hrv_fd_lf", "empatica_hrv_fd_hf",
    "empatica_hrv_fd_lf_hf_ratio", "empatica_hrv_fd_lfnu", "empatica_hrv_fd_hfnu", "empatica_hrv_fd_vlf",
    "empatica_hrv_fd_total_power",
    "eyetracker_pupil_diameter_avg", "eyetracker_pupil_change_rate", "eyetracker_blink_duration_s",
    "eyetracker_fixation_duration_s", "eyetracker_fixation_rate", "eyetracker_saccade_duration_s",
    "eyetracker_saccade_amplitude", "eyetracker_saccade_peak_velocity", "eyetracker_gaze_velocity",
    "eyetracker_gaze_dispersion", "eyetracker_distance_from_center", "eyetracker_gaze_centrality",
    "eyetracker_eyelid_aperture_avg"]
HRV_FD = OUTCOMES[6:13]
# clean_labels (H1_H3_core.R) without device prefix
LAB = {"empatica_heart_rate": "Heart rate (bpm)", "empatica_eda_tonic": "EDA tonic (SCL)", "empatica_eda_phasic": "EDA phasic (SCR)",
    "empatica_skin_temperature": "Skin temperature (°C)", "empatica_hrv_td_sdnn": "HRV SDNN (ms)", "empatica_hrv_td_rmssd": "HRV RMSSD (ms)",
    "empatica_hrv_fd_lf": "HRV LF power (ms²)", "empatica_hrv_fd_hf": "HRV HF power (ms²)", "empatica_hrv_fd_lf_hf_ratio": "HRV LF/HF ratio",
    "empatica_hrv_fd_lfnu": "HRV LF (nu)", "empatica_hrv_fd_hfnu": "HRV HF (nu)", "empatica_hrv_fd_vlf": "HRV VLF power (ms²)",
    "empatica_hrv_fd_total_power": "HRV total power (ms²)", "eyetracker_pupil_diameter_avg": "Pupil diameter (mm)",
    "eyetracker_pupil_change_rate": "Pupil change rate", "eyetracker_blink_duration_s": "Blink duration (s)",
    "eyetracker_fixation_duration_s": "Fixation duration (s)", "eyetracker_fixation_rate": "Fixation rate (/s)",
    "eyetracker_saccade_duration_s": "Saccade duration (s)", "eyetracker_saccade_amplitude": "Saccade amplitude (°)",
    "eyetracker_saccade_peak_velocity": "Saccade peak velocity (°/s)", "eyetracker_gaze_velocity": "Gaze velocity (°/s)",
    "eyetracker_gaze_dispersion": "Gaze dispersion", "eyetracker_distance_from_center": "Gaze distance from centre (px)",
    "eyetracker_gaze_centrality": "Gaze centrality", "eyetracker_eyelid_aperture_avg": "Eyelid aperture (mm)"}
SURVEYS = ["questionnaire_current_comfort_1_7", "questionnaire_pleasantness_1_5", "questionnaire_stai6_mean_1_5", "questionnaire_prs11_total_mean_1_5"]
SURVEY_LAB = {"questionnaire_current_comfort_1_7": "General comfort (1–7)", "questionnaire_pleasantness_1_5": "Pleasantness (1–5)",
    "questionnaire_stai6_mean_1_5": "State anxiety STAI-6 (1–5)", "questionnaire_prs11_total_mean_1_5": "Restorativeness PRS-11 (1–5)"}

USED = set()   # input files read, for provenance printout

RESULTS = ROOT / "output" / "statistical_analysis" / "results"

def src(rel, pc_rel):
    """pc_rel is relative to ROOT. Statistical tables must sit in the current results folder, never under archive."""
    candidate_path = ROOT.joinpath(*pc_rel.split("\\"))
    if not candidate_path.exists():
        raise FileNotFoundError(candidate_path)
    resolved = candidate_path.resolve()
    if "results" in [part.lower() for part in resolved.parts]:
        if "archive" in [part.lower() for part in resolved.parts]:
            raise RuntimeError("archived result refused: " + str(resolved))
        try:
            resolved.relative_to(RESULTS.resolve())
        except ValueError:
            raise RuntimeError("result is outside the current results folder: " + str(resolved))
    USED.add(str(resolved))
    return str(resolved)

# H1_H3_phaseavg combined outputs (output reduction 1 Nov 2026): return the rows of the former H1_Contrasts.csv /
# H3_Contrasts.csv / H2_Physio_Psycho_Concordance.csv, with the shared column names of Paper3_output_names.R.
def _reparse(t):   # filter on text, then parse like the former single-table CSV (same dtypes)
    import io
    return _read_result_csv(io.StringIO(t.to_csv(index=False)))

COMP_MAP = {
    "BikeG - BikeU": ("H1", "H1.1", "BikeU_BikeG"),
    "WalkG - WalkU": ("H1", "H1.2", "WalkU_WalkG"),
    "WalkU - BikeU": ("H3", "H3.1", "BikeU_WalkU"),
    "WalkG - BikeG": ("H3", "H3.2", "BikeG_WalkG"),
}
def h1h3_contrasts(hyp):
    results_table = _read_result_csv(src("res/H1_H3_Contrasts.csv", r"output\statistical_analysis\results\H1_H3_phaseavg\H1_H3_Contrasts.csv"), dtype=str, keep_default_na=False)
    results_table["hypothesis"] = results_table.context.str.extract(r"hypothesis=([^;]+)", expand=False)
    results_table["outcome_type"] = results_table.context.str.extract(r"outcome_type=([^;]+)", expand=False)
    results_table = results_table[(results_table.hypothesis == hyp) & (results_table.outcome_type == "Physiological")].copy()
    results_table = results_table[results_table.scale.astype(str).str.startswith("normalization=Primary")]
    sub, con = [], []
    for cmp in results_table.contrast.astype(str):
        meta = COMP_MAP.get(cmp, (None, None, None))
        sub.append(meta[1]); con.append(meta[2])
    results_table["sub_hypothesis"] = sub
    results_table["contrast"] = con
    phase_ids = results_table["contrast"].astype(str).str.split("_", n=1, expand=True)
    results_table["reference_phase_id"] = phase_ids[0]
    results_table["comparison_phase_id"] = phase_ids[1]
    results_table["normalization"] = results_table.scale.astype(str)
    results_table["adjustment_family_size"] = 26
    results_table["smd_ci_lower"] = pd.to_numeric(results_table.smd_ci_lower, errors="coerce")
    results_table["smd_ci_upper"] = pd.to_numeric(results_table.smd_ci_upper, errors="coerce")
    results_table["smd"] = pd.to_numeric(results_table.smd, errors="coerce")
    return _reparse(results_table.drop(columns=["outcome_type"], errors="ignore"))

def h2_concordance():
    results_table = _read_result_csv(src("res/H2_Concordance.csv", r"output\statistical_analysis\results\H1_H3_phaseavg\H2_Concordance.csv"), dtype=str, keep_default_na=False)
    if "test" in results_table.columns:
        results_table = results_table[results_table.test == "H2_rmcorr_physio_psycho"]
    else:
        ctx = results_table.context.fillna("").astype(str)
        if ctx.str.contains("H2_rmcorr", na=False).any():
            results_table = results_table[ctx.str.contains("H2_rmcorr", na=False)]
        else:
            results_table = results_table[ctx.str.strip() == ""]
    drop = [c for c in ("test", "mode") if c in results_table.columns]
    return _reparse(results_table.drop(columns=drop))

def md5(file_path):
    digest = hashlib.md5()
    with open(file_path, "rb") as input_stream:
        for b in iter(lambda: input_stream.read(1 << 20), b""): digest.update(b)
    return digest.hexdigest()

# ---------------- data ----------------
_cache = {}
def merged():
    if "m" in _cache: return _cache["m"]
    input_path = src("merged_all_11participants_8min.csv", r"output\merged_all_11participants_8min.csv")
    print("merged", md5(input_path)[:8], "rows pending filter")
    epochs = _read_result_csv(input_path, low_memory=False)
    epochs = epochs[epochs.ParticipantID.isin(PARTICIPANTS) & epochs.PhaseID.isin(PHASES)].copy()
    epochs["Datetime"] = pd.to_datetime(epochs["Datetime"])        # clock time as stored in the CSV (time zone not documented in the scripts)
    epochs = epochs.sort_values(["ParticipantID", "PhaseID", "Datetime"])
    epochs["t_sec"] = epochs.groupby(["ParticipantID", "PhaseID"]).Datetime.transform(lambda x: (x - x.min()).dt.total_seconds())
    # bilateral / two-sensor averages exactly as Analysis_combined_core.R average_sources (rowMeans, na.rm=TRUE)
    avg = {"atmo_pm2.5_avg": ["atmo_lys_atmotube_left__atmotube_pm2.5", "atmo_lys_atmotube_right__atmotube_pm2.5"],
           "atmo_pm1_avg": ["atmo_lys_atmotube_left__atmotube_pm1", "atmo_lys_atmotube_right__atmotube_pm1"],
           "atmo_pm10_avg": ["atmo_lys_atmotube_left__atmotube_pm10", "atmo_lys_atmotube_right__atmotube_pm10"],
           "lys_lux_avg": ["atmo_lys_LYS1__lys_lux", "atmo_lys_LYS2__lys_lux"],
           "lys_kelvin_avg": ["atmo_lys_LYS1__lys_kelvin", "atmo_lys_LYS2__lys_kelvin"]}
    for key, current_values in avg.items(): epochs[key] = epochs[current_values].mean(axis=1, skipna=True)
    assert len(epochs) == 2112 and (epochs.groupby(["ParticipantID", "PhaseID"]).size() == 48).all()
    _cache["m"] = epochs; return epochs

def ln_outcomes():
    normalization_path = src("res/Normalization_H1toH4_all_parameters.csv",
            r"output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_all_parameters.csv")
    normalization_table = _read_result_csv(normalization_path).set_index("column_name")
    tr = normalization_table.loc[OUTCOMES, "transformation_in_models"]
    assert set(tr) <= {"Natural log", "No transformation"}
    return [outcome for outcome in OUTCOMES if tr[outcome] == "Natural log"]

def analysis_units(epochs):
    """Row-level ln(x) (x<=0 -> NA) for 'Natural log' outcomes (paper3_outcome_natural_log), others native."""
    analysis_epochs = epochs.copy()
    for outcome in ln_outcomes():
        x = analysis_epochs[outcome].astype(float); analysis_epochs[outcome] = np.where(np.isfinite(x) & (x > 0), np.log(x.where(x > 0)), np.nan)
    return analysis_epochs

def phase_means(epochs, cols):
    return epochs.groupby(["ParticipantID", "PhaseID"], as_index=False)[cols].mean()

def ulab(outcome):
    return ("ln " + LAB[outcome]) if outcome in ln_outcomes() else LAB[outcome]

# ---------------- helpers ----------------
def panel_label(ax, s, dx=-0.02, dy=0.01, fig=None, size=10):
    fig = fig or ax.figure
    bb = ax.get_position()
    fig.text(bb.x0 + dx, bb.y1 + dy, s, fontsize=size, fontweight="bold", ha="right", va="bottom")

def img_axes(ax, path, title=None):
    im = Image.open(path)
    im.draft("RGB", (1100, 1100))
    im = im.convert("RGB")
    im.thumbnail((1100, 1100))
    ax.imshow(np.asarray(im)); ax.set_axis_off()
    if title: ax.set_title(title, loc="left", fontsize=8)
    return im.size

def save(fig, name):
    svg = OUT / (name + ".svg")
    png = OUT / (name + ".png")
    pdf = OUT / (name + ".pdf")
    fig.savefig(svg, format="svg", facecolor="white", bbox_inches="tight")
    fig.savefig(png, format="png", dpi=300, facecolor="white", bbox_inches="tight")
    fig.savefig(pdf, format="pdf", facecolor="white", bbox_inches="tight")
    plt.close(fig)
    print("saved", svg, png, pdf)
    try:
        from IPython.display import display, SVG
        display(SVG(filename=str(svg)))
    except Exception:
        pass

def bh(p):
    p = np.asarray(p, float); out = np.full(p.shape, np.nan); ok = np.isfinite(p); pp = p[ok]; n = len(pp)
    if n == 0: return out
    outcome = np.argsort(pp); q = pp[outcome] * n / np.arange(1, n + 1); q = np.minimum.accumulate(q[::-1])[::-1]
    r = np.empty(n); r[outcome] = np.minimum(q, 1); out[ok] = r; return out

def phase_legend(ax, loc="best", ncol=1, **kw):
    legend_handles = [Line2D([], [], marker="o", ls="", color=PHASE_COL[phase], label=PHASE_LAB[phase] + f" ({phase})", markersize=4.5) for phase in PHASES]
    return ax.legend(handles=legend_handles, loc=loc, ncol=ncol, frameon=False, **kw)

def strip_box(ax, data_by_phase, pos=None, width=0.6, log=False):
    pos = pos or list(range(len(PHASES)))
    vals = [np.asarray(current_values, float)[np.isfinite(np.asarray(current_values, float))] for current_values in data_by_phase]
    bp = ax.boxplot(vals, positions=pos, widths=width, patch_artist=True, showfliers=False, whis=(5, 95),
                    medianprops=dict(color="black", lw=0.8), boxprops=dict(lw=0.5), whiskerprops=dict(lw=0.5), capprops=dict(lw=0.5))
    for b, local_phase in zip(bp["boxes"], PHASES): b.set_facecolor(PHASE_COL[local_phase]); b.set_alpha(0.45)
    return bp


ROOT = C:\Users\sanga\Desktop\Pranav Paper3_Final
OUT  = C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures


## Study artwork

The route map, campaign timeline, sensor setup, and participant figure are separate files. They are not combined.


## Figure 1 — Exposures by phase

Ten-second epochs only. The UTCI by sound-level panels are not part of this figure.


In [2]:
# Figure 3 — exposures by phase (10 s epochs) + UTCI × dBA plane (participant phase means)
EXPOSURES = [("ucm_UTCI", "UTCI (°C)", False), ("ucm_AIR_T_mid", "Air temperature (°C)", False),
             ("ucm_MRT", "MRT (°C)", False), ("ucm_SND_dBA", "Sound level (dBA)", False),
             ("atmo_pm2.5_avg", "PM2.5 (µg/m³)", False), ("lys_lux_avg", "Illuminance (lux)", True),
             ("SVF", "Sky view factor", False), ("gvi_50m", "Green view index (%)", False)]

def fig3():
    plot_data = merged()
    fig = plt.figure(figsize=(7.2, 4.4))
    gs = gridspec.GridSpec(2, 4, figure=fig, left=0.075, right=0.985, top=0.92, bottom=0.12, hspace=0.45, wspace=0.45)
    letters = "abcdefghij"
    for index, (col, lab, logy) in enumerate(EXPOSURES):
        ax = fig.add_subplot(gs[index // 4, index % 4])
        vals = [plot_data.loc[plot_data.PhaseID == phase, col].astype(float).values for phase in PHASES]
        if logy: vals = [np.where(current_values > 0, current_values, np.nan) for current_values in vals]
        strip_box(ax, vals)
        if logy: ax.set_yscale("log")
        ax.set_xticks(range(4)); ax.set_xticklabels(PHASES, rotation=0)
        ax.set_title(lab, fontsize=7, loc="left")
        n_valid = int(np.isfinite(np.concatenate(vals)).sum())
        ax.text(0.99, 0.99, f"n={n_valid}", transform=ax.transAxes, ha="right", va="top", fontsize=5.5, color="#555555")
        panel_label(ax, letters[index], dx=-0.035)
    fig.text(0.5, 0.02, "Boxes are 10 s epochs (median, interquartile range, whiskers at the 5th and 95th percentiles). No phase means are plotted.",
             ha="center", fontsize=6, color="#444444")
    save(fig, "Figure1_exposures_by_phase")

fig3()


merged a3c4f938 rows pending filter
saved C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure1_exposures_by_phase.svg C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure1_exposures_by_phase.png C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure1_exposures_by_phase.pdf
<IPython.core.display.SVG object>


## Figure 2 — H1 + H3 forest plots (paired SMD, 26 outcomes)

Inputs (relative to `ROOT`):
* `output\statistical_analysis\results\H1_H3_phaseavg\H1_H3_Contrasts.csv` (H1 physiological rows)
* `output\statistical_analysis\results\H1_H3_phaseavg\H1_H3_Contrasts.csv` (H3 physiological rows)
* `output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_all_parameters.csv`

In [3]:
# Figure 4 â€” H1 + H3 forest plot (paired SMD, Bonett 2008; filled = primary-LMM BH q < 0.05)
def fig4():
    h1 = h1h3_contrasts("H1")
    h3 = h1h3_contrasts("H3")
    t = pd.concat([h1, h3]); t = t[t.normalization.str.contains("Primary", na=False)]
    assert len(t) == 104 and set(t.adjustment_family_size) == {26}
    panels = [("H1.1", "BikeU_BikeG", "BikeG âˆ’ BikeU", "Green âˆ’ urban, bike", PHASE_COL["BikeG"]),
              ("H1.2", "WalkU_WalkG", "WalkG âˆ’ WalkU", "Green âˆ’ urban, walk", PHASE_COL["WalkG"]),
              ("H3.1", "BikeU_WalkU", "WalkU âˆ’ BikeU", "Walk âˆ’ bike, urban", PHASE_COL["WalkU"]),
              ("H3.2", "BikeG_WalkG", "WalkG âˆ’ BikeG", "Walk âˆ’ bike, green", "#4d4d4d")]
    ypos = {outcome: index + (1.0 if index >= 13 else 0) for index, outcome in enumerate(OUTCOMES)}
    fig = plt.figure(figsize=(7.2, 5.6))
    gs = gridspec.GridSpec(1, 4, figure=fig, left=0.245, right=0.99, top=0.925, bottom=0.10, wspace=0.10)
    xl = (np.floor(t.smd_ci_lower.min()) - 0.3, np.ceil(t.smd_ci_upper.max()) + 3.2)
    for key, (sub, con, lab, ttl, col) in enumerate(panels):
        ax = fig.add_subplot(gs[0, key]); s = t[(t.sub_hypothesis == sub) & (t.contrast == con)].set_index("outcome").reindex(OUTCOMES)
        assert s.smd.notna().sum() >= 25
        ax.axvline(0, color="#777777", lw=0.6, ls="--")
        ax.axhspan(12.5, 14.5, color="white")
        for outcome in OUTCOMES:
            r = s.loc[outcome]; y = ypos[outcome]
            if not np.isfinite(r.smd): continue
            sig = bool(r.significant_adjusted)
            ax.plot([r.smd_ci_lower, r.smd_ci_upper], [y, y], color=col, lw=0.9)
            ax.plot(r.smd, y, marker="o", ms=4.2, mfc=col if sig else "white", mec=col, mew=0.9)
            if sig: ax.text(max(r.smd_ci_upper, 0) + 0.25, y, f"q={r.p_value_adjusted:.3f}" if r.p_value_adjusted >= 0.001 else f"q={r.p_value_adjusted:.0e}", fontsize=5, ha="left", va="center", color="#222222")
        ax.set_ylim(27.7, -0.7); ax.set_xlim(*xl)
        ax.set_yticks([ypos[outcome] for outcome in OUTCOMES])
        ax.set_yticklabels([ulab(outcome) for outcome in OUTCOMES] if key == 0 else [])
        ax.tick_params(axis="y", length=0 if key else 2)
        ax.set_title(f"{sub}: {lab}\n{ttl}", fontsize=7, color="black")
        ax.set_xlabel("Paired SMD (95% CI)")
        ax.grid(axis="x", color="#eeeeee", lw=0.5); ax.set_axisbelow(True)
        panel_label(ax, "abcd"[key], dx=0.0, dy=0.035)
        if key == 0:
            for (y0, y1, txt) in [(0, 12, "Empatica (13)"), (14, 26, "Eye tracker (13)")]:
                ax.annotate("", xy=(-1.28, y0 - 0.3), xytext=(-1.28, y1 + 0.3), xycoords=ax.get_yaxis_transform(), textcoords=ax.get_yaxis_transform(),
                            arrowprops=dict(arrowstyle="-", lw=0.8, color="#333333"), annotation_clip=False)
                ax.text(-1.33, (y0 + y1) / 2, txt, transform=ax.get_yaxis_transform(), rotation=90, va="center", ha="center", fontsize=7, fontweight="bold")
        ax.axhline(13.0, color="#bbbbbb", lw=0.5)
    leg = [Line2D([], [], marker="o", ls="", mfc="black", mec="black", ms=4, label="BH-FDR q < 0.05 (primary LMM, family = contrast Ã— 26 outcomes)"),
           Line2D([], [], marker="o", ls="", mfc="white", mec="black", ms=4, label="q â‰¥ 0.05")]
    fig.legend(handles=leg, loc="lower center", ncol=2, frameon=False, fontsize=6.5, bbox_to_anchor=(0.6, 0.0))
    save(fig, "Figure2_H1_H3_forest")

def fig2b_ts():
    raw = _read_result_csv(src(
        "res/H1_H3_10s.csv",
        r"output\statistical_analysis\results\H1_H3_timeseries\H1_H3_Planned_Contrasts_10s.csv",
    ))
    raw["hypothesis_id"] = raw.context.str.extract(r"hypothesis=([^;]+)", expand=False)
    t = raw[(raw.effect_type == "phase_level") & raw.hypothesis_id.isin(["H1", "H3"])].copy()
    t["std_error"] = pd.to_numeric(t.std_error, errors="coerce")
    t["estimate"] = pd.to_numeric(t.estimate, errors="coerce")
    t = t[np.isfinite(t.estimate) & np.isfinite(t.std_error) & (t.std_error < 10)]
    comp = {"BikeU_BikeG": "BikeG - BikeU", "WalkU_WalkG": "WalkG - WalkU", "BikeU_WalkU": "WalkU - BikeU", "BikeG_WalkG": "WalkG - BikeG"}
    hyp = {"H1.1": "H1", "H1.2": "H1", "H3.1": "H3", "H3.2": "H3"}
    t["lo"] = t.estimate - 1.96 * t.std_error
    t["hi"] = t.estimate + 1.96 * t.std_error
    panels = [("H1.1", "BikeU_BikeG", "BikeG − BikeU", "Green − urban, bike", PHASE_COL["BikeG"]),
              ("H1.2", "WalkU_WalkG", "WalkG − WalkU", "Green − urban, walk", PHASE_COL["WalkG"]),
              ("H3.1", "BikeU_WalkU", "WalkU − BikeU", "Walk − bike, urban", PHASE_COL["WalkU"]),
              ("H3.2", "BikeG_WalkG", "WalkG − BikeG", "Walk − bike, green", "#4d4d4d")]
    ypos = {outcome: index + (1.0 if index >= 13 else 0) for index, outcome in enumerate(OUTCOMES)}
    fig = plt.figure(figsize=(7.2, 5.6))
    gs = gridspec.GridSpec(1, 4, figure=fig, left=0.245, right=0.99, top=0.90, bottom=0.12, wspace=0.10)
    for key, (sub, con, lab, ttl, col) in enumerate(panels):
        ax = fig.add_subplot(gs[0, key])
        s = t[(t.hypothesis_id == hyp[sub]) & (t.contrast == comp[con])].drop_duplicates("outcome").set_index("outcome").reindex(OUTCOMES)
        ax.axvline(0, color="#777777", lw=0.6, ls="--")
        ax.axhspan(12.5, 14.5, color="white")
        for outcome in OUTCOMES:
            r = s.loc[outcome]; y = ypos[outcome]
            if not np.isfinite(r.estimate):
                continue
            sig = bool(r.significant_adjusted) if pd.notna(r.significant_adjusted) else False
            lo, hi = np.clip(r.lo, -4, 4), np.clip(r.hi, -4, 4)
            ax.plot([lo, hi], [y, y], color=col, lw=0.9)
            ax.plot(float(np.clip(r.estimate, -4, 4)), y, marker="o", ms=4.2, mfc=col if sig else "white", mec=col, mew=0.9)
        ax.set_ylim(27.7, -0.7); ax.set_xlim(-4.2, 4.2)
        ax.set_yticks([ypos[o] for o in OUTCOMES])
        ax.set_yticklabels([ulab(o) for o in OUTCOMES] if key == 0 else [])
        ax.set_title(f"{sub}: {lab}\n{ttl}", fontsize=7)
        ax.set_xlabel("10 s coefficient (z scale)")
        panel_label(ax, "abcd"[key], dx=0.0, dy=0.035)
        ax.axhline(13.0, color="#bbbbbb", lw=0.5)
    fig.suptitle("10 s phase-level contrasts (non-degenerate fits; intervals clipped at ±4)", fontsize=8, y=0.98)
    save(fig, "Figure2b_H1_H3_forest_10s")

fig4()
fig2b_ts()

saved C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure2_H1_H3_forest.svg C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure2_H1_H3_forest.png C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure2_H1_H3_forest.pdf
<IPython.core.display.SVG object>
saved C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure2b_H1_H3_forest_10s.svg C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure2b_H1_H3_forest_10s.png C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure2b_H1_H3_forest_10s.pdf
<IPython.core.display.SVG object>


## Figure 3 — H2 physiology × psychology rmcorr

Inputs (relative to `ROOT`):
* `output\statistical_analysis\results\H1_H3_phaseavg\H2_Concordance.csv` (test = H2_rmcorr_physio_psycho)
* `output\merged_all_11participants_8min.csv`

In [4]:
# Figure 3 — H2 rmcorr heatmap (all questionnaire × physiological outcomes)
def fig5():
    h2 = h2_concordance()
    assert len(h2) > 0, "H2 concordance table is empty after filtering pooled rows"
    surveys = sorted(c for c in h2.outcome.drop_duplicates() if str(c).startswith("questionnaire"))
    R = h2.pivot(index="outcome", columns="term", values="correlation").reindex(index=surveys, columns=OUTCOMES)
    Q = h2.pivot(index="outcome", columns="term", values="p_value_adjusted").reindex(index=surveys, columns=OUTCOMES)

    def qlab(col):
        if col in SURVEY_LAB:
            return SURVEY_LAB[col]
        return col.replace("questionnaire_", "").replace("_mean_1_5", "").replace("_1_7", "").replace("_1_5", "").replace("_", " ")

    n_surv = len(surveys)
    fig_h = max(7.2, 0.26 * n_surv + 2.6)
    fig = plt.figure(figsize=(7.4, fig_h))
    ax = fig.add_axes([0.22, 0.16, 0.66, 0.76])
    im = ax.imshow(R.values, cmap=plt.get_cmap("RdBu_r"), vmin=-0.8, vmax=0.8, aspect="auto")
    for i in range(R.shape[0]):
        for j in range(R.shape[1]):
            v = R.values[i, j]
            q = Q.values[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.2f}".replace("0.", ".").replace("-.", "−."), ha="center", va="center", fontsize=4.6,
                        color="white" if abs(v) > 0.5 else "black", fontweight="bold" if np.isfinite(q) and q < 0.05 else "normal")
            if np.isfinite(q) and q < 0.05:
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="black", lw=1.3))
    ax.set_yticks(range(n_surv))
    ax.set_yticklabels([qlab(s) for s in surveys], fontsize=6.5)
    ax.set_xticks(range(26))
    ax.set_xticklabels([ulab(o) for o in OUTCOMES], rotation=55, ha="left", rotation_mode="anchor", fontsize=5.6)
    ax.xaxis.tick_top()
    ax.tick_params(axis="x", length=0, pad=2)
    ax.tick_params(axis="y", length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.axvline(12.5, color="black", lw=0.8)
    cax = fig.add_axes([0.895, 0.16, 0.015, 0.76])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("rmcorr r", fontsize=6.5)
    cb.ax.tick_params(labelsize=5.5)
    fig.text(0.22, 0.06,
             "Black box / bold = BH-FDR q < 0.05. Pooled rmcorr on 44 participant × phase means (not 10 s epochs). Vertical line: Empatica | eye tracker.",
             fontsize=5.8, color="#333333")
    panel_label(ax, "a", dx=-0.12, dy=0.02)
    save(fig, "Figure3_H2_rmcorr")

fig5()

saved C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure3_H2_rmcorr.svg C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure3_H2_rmcorr.png C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure3_H2_rmcorr.pdf
<IPython.core.display.SVG object>


## Figure 4 — Sound composition by phase

Panel a is the supplied score-versus-proportion figure. Unidentified sound and the broad frequency-band panel are omitted.


In [5]:
# Figure 4 - sound composition by phase (10 s epochs)
# (a) dBA per epoch with participant means, (b) octave-band profile, (c) YAMNet category mean proportion,
# (d) YAMNet category mean score with 0.20 threshold, (e) example participant 10 s dBA time series.
SND_CATS = [("wind", "Wind"), ("rural_natural", "Rural/Natural"), ("vehicle", "Vehicle"),
            ("urban_manmade_neutral", "Urban neutral"), ("urban_manmade_positive", "Urban positive"),
            ("urban_manmade_negative", "Urban negative"), ("mechanisms", "Mechanisms"), ("unidentify", "Unidentified")]

def fig4_sound():
    snd = merged()
    snd = snd.assign(t_min=snd["t_sec"] / 60.0)
    dba = "ucm_SND_dBA"
    prop_cols = [(k + "_proportion", lab) for k, lab in SND_CATS if k + "_proportion" in snd.columns]
    score_cols = [(k + "_score", lab) for k, lab in SND_CATS if k + "_score" in snd.columns]
    oc = [c for c in snd.columns if c.startswith("octave_proportion_")]
    n_pid = {ph: snd.loc[snd.PhaseID == ph, "ParticipantID"].nunique() for ph in PHASES}

    def color_ticks(ax):
        for tl, ph in zip(ax.get_xticklabels(), PHASES):
            tl.set_color(PHASE_COL[ph])

    fig, axd = plt.subplot_mosaic([["a", "b"], ["c", "d"], ["e", "e"]], figsize=(7.2, 8.6), height_ratios=[1, 1.05, 0.85])
    fig.subplots_adjust(left=0.13, right=0.98, top=0.955, bottom=0.085, hspace=0.62, wspace=0.5)

    # (a) dBA distribution per phase + participant phase means
    ax = axd["a"]
    vals = [snd.loc[snd.PhaseID == ph, dba].astype(float).dropna().values for ph in PHASES]
    vp = ax.violinplot(vals, positions=range(4), widths=0.8, showextrema=False)
    for b, ph in zip(vp["bodies"], PHASES):
        b.set_facecolor(PHASE_COL[ph]); b.set_alpha(0.3); b.set_edgecolor("none")
    bp = strip_box(ax, vals, width=0.25)
    for b in bp["boxes"]:
        b.set_alpha(0.55)
    pm = snd.groupby(["ParticipantID", "PhaseID"])[dba].mean().reset_index()
    rng = np.random.default_rng(1)
    for i, ph in enumerate(PHASES):
        y = pm.loc[pm.PhaseID == ph, dba].values
        ax.scatter(i + 0.22 + rng.uniform(-0.04, 0.04, len(y)), y, s=7, color=PHASE_COL[ph], edgecolor="black", lw=0.3, zorder=3)
        ax.text(i, 1.01, f"mean {np.nanmean(vals[i]):.1f}", transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=5.4)
    ax.set_xticks(range(4)); ax.set_xticklabels([f"{ph}\n(n={n_pid[ph]})" for ph in PHASES]); color_ticks(ax)
    ax.set_ylabel("Sound level (dBA)")
    ax.set_title("A-weighted level: 10 s epochs, dots = participant means", fontsize=6.6, loc="left", pad=10)

    # (b) octave-band profile
    ax = axd["b"]
    fc = [float(c.replace("octave_proportion_", "").replace("_Hz", "").replace("_", ".")) for c in oc]
    for ph in PHASES:
        x = snd.loc[snd.PhaseID == ph, oc]
        ax.plot(fc, x.mean().values * 100, "-o", color=PHASE_COL[ph], ms=2.5, lw=1, label=ph)
    ax.set_xscale("log"); ax.set_xticks(fc)
    ax.set_xticklabels([("%g" % f) if f < 1000 else ("%gk" % (f / 1000)) for f in fc], fontsize=5.6)
    ax.minorticks_off(); ax.set_xlabel("Octave-band centre frequency (Hz)"); ax.set_ylabel("Mean energy share (%)")
    ax.set_title("Octave-band profile", fontsize=7, loc="left"); ax.legend(frameon=False, fontsize=5.2)

    # (c) mean category proportion, phase x category heatmap
    ax = axd["c"]
    P = np.array([[snd.loc[snd.PhaseID == ph, c].mean() for c, _ in prop_cols] for ph in PHASES])
    im = ax.imshow(P.T, aspect="auto", cmap="YlOrRd", vmin=0, vmax=max(0.01, np.nanmax(P)))
    for i in range(P.shape[0]):
        for j in range(P.shape[1]):
            ax.text(i, j, f"{P[i, j]:.2f}", ha="center", va="center", fontsize=5.4,
                    color="white" if P[i, j] > 0.6 * np.nanmax(P) else "black")
    ax.set_xticks(range(4)); ax.set_xticklabels(PHASES); color_ticks(ax)
    ax.set_yticks(range(len(prop_cols))); ax.set_yticklabels([lab for _, lab in prop_cols], fontsize=5.8)
    for s in ax.spines.values():
        s.set_visible(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.03); cb.ax.tick_params(labelsize=5.5); cb.outline.set_linewidth(0.4)
    ax.set_title("YAMNet category: mean proportion of frames", fontsize=6.6, loc="left")

    # (d) mean category score per phase with 0.20 threshold
    ax = axd["d"]; w = 0.2
    S = np.array([[snd.loc[snd.PhaseID == ph, c].mean() for c, _ in score_cols] for ph in PHASES])
    for i, ph in enumerate(PHASES):
        ax.barh(np.arange(len(score_cols)) + (i - 1.5) * w, S[i], height=w, color=PHASE_COL[ph], label=ph)
    ax.axvline(0.20, color="black", ls="--", lw=0.7)
    ax.text(0.20, -0.75, " 0.20 threshold", fontsize=5.4, va="center", ha="left")
    ax.set_yticks(range(len(score_cols))); ax.set_yticklabels([lab for _, lab in score_cols], fontsize=5.8); ax.invert_yaxis()
    ax.set_xlabel("Mean YAMNet score")
    ax.set_title("YAMNet category: mean score", fontsize=6.6, loc="left")
    ax.legend(frameon=False, fontsize=5.4, loc="lower right")

    # (e) example dBA time series of one participant per phase
    ax = axd["e"]
    cnt = snd.dropna(subset=[dba]).groupby("ParticipantID").PhaseID.nunique()
    ex_pid = "P10" if cnt.get("P10", 0) == 4 else cnt.idxmax()
    for ph in PHASES:
        d = snd[(snd.ParticipantID == ex_pid) & (snd.PhaseID == ph)]
        ax.plot(d.t_min, d[dba], "-o", ms=1.8, lw=0.9, color=PHASE_COL[ph], label=f"{ph} (mean {d[dba].mean():.1f})")
    ax.set_xlim(0, 8); ax.set_xticks(range(9))
    ax.set_xlabel("Time in phase (min)"); ax.set_ylabel("Sound level (dBA)")
    ax.set_title(f"Example participant {ex_pid}: 10 s dBA across the 8-min phase", fontsize=7, loc="left")
    ax.legend(frameon=False, fontsize=5.6, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.22))

    for k, a in axd.items():
        panel_label(a, k, dx=-(0.075 if k in ("a", "e") else 0.105 if k == "c" else 0.06), dy=0.012)
    fig.text(0.13, 0.005, "Pooled 10 s epochs from merged_all_11participants_8min.csv; n = participants per phase. "
             "Boxes: median, IQR, 5-95 % whiskers.", fontsize=5.6, color="#444444")
    print("mean dBA:", {ph: round(float(np.nanmean(v)), 1) for ph, v in zip(PHASES, vals)})
    save(fig, "Figure4_sound_composition")

fig4_sound()


mean dBA: {'BikeU': 71.4, 'WalkU': 67.5, 'BikeG': 60.3, 'WalkG': 61.6}
saved C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure4_sound_composition.svg C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure4_sound_composition.png C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure4_sound_composition.pdf
<IPython.core.display.SVG object>


## Figure 5 — Participants and data coverage

Inputs (relative to `ROOT`):
* `Figures\Final_manuscript_figures\Participants matched_8min.png`
* `output\merged_all_11participants_8min.csv`

In [6]:
# Figure 7 — participants and data coverage
def figS1():
    plot_data = merged(); a = analysis_units(plot_data)
    photos = src("art/Participants matched_8min.png", r"Figures\Final_manuscript_figures\Participants matched_8min.png")
    fig = plt.figure(figsize=(7.2, 9.6))
    axa = fig.add_axes([0.06, 0.775, 0.90, 0.215]); img_axes(axa, photos); panel_label(axa, "a", dx=-0.01, dy=-0.01)
    # b: session timeline
    axb = fig.add_axes([0.09, 0.575, 0.88, 0.165])
    order = sorted(PARTICIPANTS, key=lambda participant_id: plot_data[plot_data.ParticipantID == participant_id].Datetime.min().date())
    for index, pid in enumerate(order):
        x = plot_data[plot_data.ParticipantID == pid]; day = x.Datetime.min().normalize()
        for ph in PHASES:
            y = x[x.PhaseID == ph]; t0 = (y.Datetime.min() - day).total_seconds() / 3600; t1 = (y.Datetime.max() - day).total_seconds() / 3600 + 10 / 3600
            axb.barh(index, t1 - t0, left=t0, color=PHASE_COL[ph], height=0.7)
        first = x.sort_values("Datetime").PhaseID.iloc[0]
        axb.text(16.15, index, x.Datetime.min().strftime("%d %b") + ("  green first" if first in ("BikeG", "WalkG") else "  urban first"), va="center", fontsize=5.4)
    axb.set_yticks(range(len(order))); axb.set_yticklabels(order, fontsize=6); axb.invert_yaxis()
    axb.set_xlim(12, 17.2); axb.set_xticks(range(12, 17)); axb.set_xticklabels([f"{h}:00" for h in range(12, 17)])
    axb.set_xlabel("Clock time of 10 s epochs as stored in the merged file (time zone not documented)", fontsize=6.5)
    phase_legend(axb, loc="lower left", ncol=4, fontsize=5.6, bbox_to_anchor=(0.0, 1.0))
    panel_label(axb, "b", dx=-0.05, dy=0.03)
    # c: outcome coverage
    cells = [(participant_id, ph) for participant_id in PARTICIPANTS for ph in PHASES]
    C = np.array([[a[(a.ParticipantID == participant_id) & (a.PhaseID == ph)][outcome].astype(float).notna().mean() * 100 for participant_id, ph in cells] for outcome in OUTCOMES])
    axc = fig.add_axes([0.21, 0.255, 0.70, 0.25])
    im = axc.imshow(C, cmap="viridis", vmin=0, vmax=100, aspect="auto", interpolation="nearest")
    axc.set_yticks(range(26)); axc.set_yticklabels([ulab(outcome) for outcome in OUTCOMES], fontsize=5.2)
    axc.set_xticks(range(len(cells))); axc.set_xticklabels([ph for _, ph in cells], rotation=90, fontsize=4.2)
    for t, (_, ph) in zip(axc.get_xticklabels(), cells): t.set_color(PHASE_COL[ph])
    for key, participant_id in enumerate(PARTICIPANTS):
        axc.text(key * 4 + 1.5, -0.9, participant_id, ha="center", va="bottom", fontsize=5.4)
        if key: axc.axvline(key * 4 - 0.5, color="white", lw=0.8)
    axc.axhline(12.5, color="white", lw=0.8); axc.tick_params(length=0)
    cb = fig.colorbar(im, cax=fig.add_axes([0.925, 0.255, 0.012, 0.25])); cb.set_label("% of the 48 epochs with a valid value", fontsize=6); cb.ax.tick_params(labelsize=5.5)
    panel_label(axc, "c", dx=-0.17, dy=0.02)
    # d: exposure coverage
    expo = [("ucm_UTCI", "UTCI"), ("ucm_AIR_T_mid", "Air T mid"), ("ucm_MRT", "MRT"), ("ucm_SUN_Gh", "Solar Gh"), ("ucm_SND_dBA", "dBA"),
            ("spectral_centroid_Hz", "Sound spectrum"), ("vehicle_proportion", "YAMNet"), ("atmo_pm2.5_avg", "PM2.5 (L/R mean)"),
            ("lys_lux_avg", "Lux (LYS mean)"), ("lys_kelvin_avg", "Colour T (LYS mean)"), ("SVF", "SVF"), ("gvi_50m", "GVI 50 m")]
    E = np.array([[plot_data[(plot_data.ParticipantID == participant_id) & (plot_data.PhaseID == ph)][c].astype(float).notna().mean() * 100 for participant_id, ph in cells] for c, _ in expo])
    axd = fig.add_axes([0.21, 0.07, 0.70, 0.125])
    axd.imshow(E, cmap="viridis", vmin=0, vmax=100, aspect="auto", interpolation="nearest")
    axd.set_yticks(range(len(expo))); axd.set_yticklabels([l for _, l in expo], fontsize=5.4)
    axd.set_xticks(range(len(cells))); axd.set_xticklabels([ph for _, ph in cells], rotation=90, fontsize=4.2)
    for t, (_, ph) in zip(axd.get_xticklabels(), cells): t.set_color(PHASE_COL[ph])
    for key, participant_id in enumerate(PARTICIPANTS):
        axd.text(key * 4 + 1.5, -0.7, participant_id, ha="center", va="bottom", fontsize=5.4)
        if key: axd.axvline(key * 4 - 0.5, color="white", lw=0.8)
    axd.tick_params(length=0)
    for index in range(len(expo)):
        for column_index in range(len(cells)):
            if E[index, column_index] < 99.5: axd.text(column_index, index, f"{E[index, column_index]:.0f}", ha="center", va="center", fontsize=3.6, color="white" if E[index, column_index] < 60 else "black")
    panel_label(axd, "d", dx=-0.17, dy=0.025)
    hrv_av = (plot_data["empatica_hrv_fd_5min_available"].astype(str).str.lower().isin(["true", "1"])).mean() * 100
    fig.text(0.06, 0.006, f"c: ln outcomes counted valid only if > 0 (no EDA tonic value is ≤ 0; gaps are missing values). Eye-tracker: no data for P13 (all phases) or P8 WalkU.\nH4 additionally masks the 7 HRV frequency-domain outcomes where the 5-min window flag is false ({hrv_av:.1f} % of epochs flagged available). d: values printed where coverage < 99.5 %.",
             fontsize=5.3, color="#444444")
    print("S1 coverage min outcome", np.nanmin(C), "min exposure", np.nanmin(E))
    save(fig, "Figure5_participants_coverage")

figS1()

S1 coverage min outcome 0.0 min exposure 87.5
saved C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure5_participants_coverage.svg C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure5_participants_coverage.png C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure5_participants_coverage.pdf
<IPython.core.display.SVG object>


## Figure 6 — 26 outcomes by phase (participant × phase means)

Inputs (relative to `ROOT`):
* `output\merged_all_11participants_8min.csv`
* `output\statistical_analysis\results\H1_H3_phaseavg\H1_H3_Contrasts.csv` (H1 and H3 physiological rows)

In [7]:
# Figure 8 — 26 outcomes by phase (participant × phase means, analysis units)
def figS2():
    plot_data = analysis_units(merged()); pm = phase_means(plot_data, OUTCOMES)
    h1 = h1h3_contrasts("H1")
    h3 = h1h3_contrasts("H3")
    t = pd.concat([h1, h3]); t = t[t.normalization.str.startswith("Primary")]
    pos = {phase: index for index, phase in enumerate(PHASES)}
    fig, axes = plt.subplots(6, 5, figsize=(7.2, 9.6))
    fig.subplots_adjust(left=0.07, right=0.99, top=0.975, bottom=0.04, hspace=0.62, wspace=0.52)
    letters = "abcdefghijklmnopqrstuvwxyz"
    for key, outcome in enumerate(OUTCOMES):
        ax = axes.flat[key]
        w = pm.pivot(index="ParticipantID", columns="PhaseID", values=outcome).reindex(columns=PHASES)
        for pid, r in w.iterrows(): ax.plot(range(4), r.values, color="#bbbbbb", lw=0.4, zorder=1)
        for phase in PHASES:
            current_values = w[phase].dropna().values
            ax.scatter(np.full(len(current_values), pos[phase]), current_values, s=6, color=PHASE_COL[phase], zorder=2, lw=0)
            ax.errorbar(pos[phase] + 0.22, current_values.mean(), yerr=current_values.std(ddof=1) / np.sqrt(len(current_values)), fmt="s", ms=2.8, color="black", lw=0.7, capsize=1.2, zorder=3)
        for _, r in t[t.outcome == outcome].iterrows():
            if "reference_mean_raw" in t.columns and pd.notna(r.reference_mean_raw):
                assert abs(w[r.reference_phase_id].mean() - r.reference_mean_raw) <= 1e-6 * max(1, abs(r.reference_mean_raw))
        sig = t[(t.outcome == outcome) & (t.significant_adjusted.astype(str).str.upper() == "TRUE")]
        lo, hi = np.nanmin(w.values), np.nanmax(w.values); rng = hi - lo if hi > lo else 1; lev = 0
        for _, r in sig.sort_values("contrast").iterrows():
            a_, b_ = sorted([pos[r.reference_phase_id], pos[r.comparison_phase_id]]); y = hi + rng * (0.08 + 0.13 * lev)
            ax.plot([a_, a_, b_, b_], [y - rng * 0.03, y, y, y - rng * 0.03], color="black", lw=0.6)
            ax.text((a_ + b_) / 2, y, "*" if r.p_value_adjusted >= 0.01 else ("**" if r.p_value_adjusted >= 0.001 else "***"), ha="center", va="bottom", fontsize=6.5, linespacing=0.6)
            lev += 1
        ax.set_ylim(lo - rng * 0.06, hi + rng * (0.12 + 0.13 * max(lev, 0.4)))
        ax.set_xticks(range(4)); ax.set_xticklabels(PHASES, fontsize=4.8, rotation=0)
        for tl, phase in zip(ax.get_xticklabels(), PHASES): tl.set_color(PHASE_COL[phase])
        ax.tick_params(axis="y", labelsize=5); ax.set_xlim(-0.45, 3.6)
        ax.set_title(ulab(outcome), fontsize=5.8, pad=2)
        ax.text(-0.33, 1.06, letters[key], transform=ax.transAxes, fontsize=8, fontweight="bold", va="bottom")
    for ax in axes.flat[26:]: ax.set_axis_off()
    axl = axes.flat[26]
    txt = ("Grey lines / coloured dots: participant × phase means\n(ln applied per 10 s epoch for ln outcomes).\n"
           "Black square: mean ± SE across participants.\n"
           "Brackets: primary H1.1 (BikeU–BikeG), H1.2 (WalkU–WalkG),\nH3.1 (BikeU–WalkU), H3.2 (BikeG–WalkG) contrasts with\n"
           "BH q < 0.05 (* q<0.05, ** q<0.01, *** q<0.001;\nfamily = 26 outcomes per contrast).\n"
           "Eye-tracker panels: 10 participants (no eye data for P13);\nWalkU 9 (P8 WalkU missing).")
    axl.text(0.0, 1.0, txt, transform=axl.transAxes, fontsize=5.6, va="top")
    save(fig, "Figure6_outcomes_by_phase")

figS2()

saved C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure6_outcomes_by_phase.svg C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure6_outcomes_by_phase.png C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Figure6_outcomes_by_phase.pdf
<IPython.core.display.SVG object>


## Input files read in this session

In [8]:
for source_path in sorted(USED): print(source_path)

C:\Users\sanga\Desktop\Pranav Paper3_Final\Figures\Final_manuscript_figures\Participants matched_8min.png
C:\Users\sanga\Desktop\Pranav Paper3_Final\output\merged_all_11participants_8min.csv
C:\Users\sanga\Desktop\Pranav Paper3_Final\output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_all_parameters.csv
C:\Users\sanga\Desktop\Pranav Paper3_Final\output\statistical_analysis\results\H1_H3_phaseavg\H1_H3_Contrasts.csv
C:\Users\sanga\Desktop\Pranav Paper3_Final\output\statistical_analysis\results\H1_H3_phaseavg\H2_Concordance.csv
C:\Users\sanga\Desktop\Pranav Paper3_Final\output\statistical_analysis\results\H1_H3_timeseries\H1_H3_Planned_Contrasts_10s.csv


## Sound segment figure: Welch PSD (Di bands) + YAMNet seven researcher categories

In [9]:
# Sound segment figure: Welch PSD (Di bands) + YAMNet seven researcher categories
# Ported from scripts/Sound_yamnet_seven_category_figure.py (same logic; ASCII labels).
# Self-contained: needs soundfile, tensorflow, tensorflow_hub, scipy, matplotlib.
# Panels: 1 waveform, 2 Welch PSD with Di broad bands, 3 YAMNet log-mel spectrogram,
#         4 per-patch heatmap of the 7 categories, 5 row-level *_score / *_proportion bars.
import csv as _snd_csv
import io as _snd_io
import math as _snd_math
from pathlib import Path as _SndPath

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import trapezoid
from scipy.signal import resample_poly, welch

# ---------------- config ----------------
# Change WAV_PATH to the phase WAV you want to plot (one participant-phase recording).
# The raw UCM phase WAVs are not stored in this project folder; this default is the raw
# UCM root used by the YAMNet notebook. If WAV_PATH is a folder, the first *.wav found
# in it (sorted, recursive) is used.
WAV_PATH = _SndPath(r"C:\Users\pandya\OneDrive - UCL\Field experiment raw data\Complete Participantwise data\Final Data\ucm")
SECONDS = 10
_SND_FALLBACK_ROOT = _SndPath(r"C:\Users\sanga\Desktop\Pranav Paper3_Final")
_SND_ROOT = next(
    (r for r in (_SndPath.cwd().parent, _SndPath.cwd(), _SND_FALLBACK_ROOT)
     if (r / "output" / "merged_all_11participants_8min.csv").exists()),
    _SND_FALLBACK_ROOT,
)
OUTPUT = _SND_ROOT / "Figures" / "Figure_sound_yamnet_segment.png"
YAMNET_URL = "https://tfhub.dev/google/yamnet/1"

# ---------------- optional heavy dependencies ----------------
_SND_MISSING = []
try:
    import soundfile as sf
except Exception as _e:
    sf = None
    _SND_MISSING.append("soundfile (" + type(_e).__name__ + ")")
try:
    import tensorflow as tf
except Exception as _e:
    tf = None
    _SND_MISSING.append("tensorflow (" + type(_e).__name__ + ")")
try:
    import tensorflow_hub as hub
except Exception as _e:
    hub = None
    _SND_MISSING.append("tensorflow_hub (" + type(_e).__name__ + ")")

# ---------------- constants (identical to the script) ----------------
YAMNET_SR = 16_000
PATCH_WINDOW_S = 0.96
PATCH_HOP_S = 0.48
AUDIO_MIN_HZ = 20.0
AUDIO_MAX_HZ = 20_000.0
DETECTION_THRESHOLD = 0.20

DI_BROAD_BANDS = (
    ("20-200 Hz", 20.0, 200.0, "#9467bd"),
    ("200-2000 Hz", 200.0, 2000.0, "#2ca02c"),
    ("2000-20000 Hz", 2000.0, 20_000.0, "#ff7f0e"),
)

RESEARCHER_CATEGORIES = {
    "Wind": "Wind|Wind noise (microphone)|Howl|Whoosh, swoosh, swish|Gust of wind|Blowing (wind)|Storm|Hurricane|Tornado".split(
        "|"
    ),
    "Rural/Natural": "Outside, rural or natural|Forest|Bird|Bird vocalization, bird call, bird song|Chirp, tweet|Coo|Crow|Insect|Cricket|Bee, wasp, etc.|Frog|Nature sounds|Stream|Water|Rain|Thunderstorm|Thunder|Ocean|Waves, surf|Animal|Livestock, farm animals, working animals|Horse|Cattle, bovinae|Sheep|Goat|Pig|Chicken, rooster|Dog|Cat|Wild animals|Wolf|Bird flight, flapping wings".split(
        "|"
    ),
    "Vehicle": "Vehicle|Motor vehicle (road)|Car|Car passing by|Truck|Bus|Motorcycle|Emergency vehicle|Siren|Police car (siren)|Ambulance (siren)|Fire engine, fire truck (siren)|Aircraft|Airplane|Helicopter|Train|Rail transport|Train horn|Subway, metro, underground|Boat, Water vehicle|Ship|Motorboat, speedboat|Engine|Engine starting|Idling|Accelerating, revving, vroom|Car alarm|Vehicle horn, car horn, honking".split(
        "|"
    ),
    "Urban/Manmade_neutral": "Outside, urban or manmade|City|Urban|Footsteps|Walking|Running|Skateboard|Bicycle".split("|"),
    "Urban/Manmade_positive": "Crowd|Cheering|Applause|Chatter|Conversation|Speech|Human voice|Street music|Busker|Music".split(
        "|"
    ),
    "Urban/Manmade_negative": "Traffic noise, roadway noise|Construction site|Jackhammer|Bulldozer|Excavator|Drilling|Hammering|Sawing|Power tool|Chainsaw|Siren|Emergency vehicle|Industrial noise|Factory|Machinery|Air conditioning".split(
        "|"
    ),
    "Mechanisms": "Mechanisms|Tools|Hammer|Hammering|Wood|Sawing|Filing (rasp)|Sanding|Power tool|Drill|Jackhammer|Chainsaw|Medium engine (mid frequency)|Light engine (high frequency)|Heavy engine (low frequency)|Engine knocking|Engine starting|Idling|Accelerating, revving, vroom|Door|Sliding door|Slam|Knock|Tap|Squeak|Creak|Ratchet, pawl|Clock|Tick|Ticking|Mechanical fan|Air conditioning|Microwave oven|Blender|Water tap, faucet|Flush|Zipper (clothing)|Keys jangling|Coin (dropping)|Scissors|Electric shaver, electric razor|Lawn mower|Printing|Typewriter|Keyboard (typing)".split(
        "|"
    ),
}

CATEGORY_ORDER = list(RESEARCHER_CATEGORIES.keys())


# ---------------- functions (ported from the script) ----------------
def load_class_names(model):
    class_map_path = model.class_map_path().numpy().decode("utf-8")
    with tf.io.gfile.GFile(class_map_path, "r") as handle:
        rows = list(_snd_csv.reader(_snd_io.StringIO(handle.read())))
    names = [row[2] for row in rows if len(row) >= 3 and row[0] != "index"]
    if len(names) != 521:
        raise RuntimeError(f"Expected 521 YAMNet classes, got {len(names)}")
    return names


def to_mono(waveform):
    data = np.asarray(waveform, dtype=np.float32)
    if data.ndim == 2:
        data = data[:, 0]
    return np.clip(data, -1.0, 1.0)


def to_mono_16k(waveform, sr):
    data = to_mono(waveform)
    if sr == YAMNET_SR:
        return data
    g = _snd_math.gcd(int(sr), YAMNET_SR)
    return resample_poly(data, YAMNET_SR // g, int(sr) // g).astype(np.float32)


def category_indices(class_names):
    mapping = {}
    for category, labels in RESEARCHER_CATEGORIES.items():
        label_set = set(labels)
        idx = np.array([i for i, name in enumerate(class_names) if name in label_set], dtype=int)
        if idx.size == 0:
            raise RuntimeError(f"No YAMNet labels matched category {category}")
        mapping[category] = idx
    return mapping


def patch_category_matrix(frame_scores, cat_idx):
    rows = []
    for category in CATEGORY_ORDER:
        idx = cat_idx[category]
        rows.append(frame_scores[:, idx].sum(axis=1))
    return np.stack(rows, axis=1)


def _integrate_psd(freqs, psd, low, high):
    interior = freqs[(freqs > low) & (freqs < high)]
    pts = np.concatenate(([low], interior, [high]))
    return float(trapezoid(np.interp(pts, freqs, psd), pts))


def welch_psd(segment, sr):
    """Match notebook spectral_features Welch branch (Di proportions only)."""
    props = {label: float("nan") for label, _, _, _ in DI_BROAD_BANDS}
    max_hz = min(AUDIO_MAX_HZ, sr / 2.0)
    if segment.size < max(32, int(0.25 * sr)) or max_hz <= AUDIO_MIN_HZ or not np.any(np.abs(segment) > 0):
        return np.array([]), np.array([]), props

    nperseg = min(len(segment), max(1024, int(sr)))
    freqs, psd = welch(
        segment.astype(np.float64),
        fs=float(sr),
        window="hann",
        nperseg=nperseg,
        noverlap=nperseg // 2,
        detrend="constant",
        scaling="density",
    )
    valid = (freqs >= AUDIO_MIN_HZ) & (freqs <= max_hz) & np.isfinite(psd)
    f, p = freqs[valid], psd[valid]
    if f.size < 2 or not np.any(p > 0):
        return np.array([]), np.array([]), props

    total = float(trapezoid(p, f))
    if not np.isfinite(total) or total <= 0:
        return f, p, props

    for label, low, high, _ in DI_BROAD_BANDS:
        if high <= max_hz and low >= AUDIO_MIN_HZ:
            e = _integrate_psd(f, p, low, high)
            props[label] = float(e / total) if np.isfinite(e) else float("nan")
    return f, p, props


def rollup_row_metrics(frame_scores, class_names, threshold=DETECTION_THRESHOLD):
    """Same logic as classify_researcher_category in the YAMNet notebook."""
    class_means = frame_scores.mean(axis=0)
    category_scores = {}
    category_proportions = {c: 0.0 for c in CATEGORY_ORDER}
    category_proportions["Unidentify"] = 0.0
    any_match = np.zeros(frame_scores.shape[0], dtype=bool)

    for category, labels in RESEARCHER_CATEGORIES.items():
        label_set = set(labels)
        indices = [i for i, label in enumerate(class_names) if label in label_set]
        category_scores[category] = float(class_means[indices].sum())
        if indices:
            active = np.any(frame_scores[:, indices] >= threshold, axis=1)
            category_proportions[category] = float(active.mean())
            any_match |= active

    best = max(category_scores, key=category_scores.get)
    if category_scores[best] < threshold or category_scores[best] == 0:
        best = "Unidentify"
    category_proportions["Unidentify"] = float((~any_match).mean())
    return category_scores, category_proportions, best


def make_sound_segment_figure(wav_path, seconds, output):
    wav_path = _SndPath(wav_path).expanduser().resolve()
    if not wav_path.is_file():
        raise FileNotFoundError(wav_path)

    audio, sr = sf.read(str(wav_path), always_2d=False)
    n_samples = int(min(len(audio), max(1, seconds) * sr))
    segment = to_mono(np.asarray(audio[:n_samples]))
    seg16 = to_mono_16k(segment, int(sr))

    freqs, psd, di_props = welch_psd(segment, int(sr))

    model = hub.load(YAMNET_URL)
    class_names = load_class_names(model)
    cat_idx = category_indices(class_names)

    scores, _, spectrogram = model(tf.constant(seg16, dtype=tf.float32))
    frame_scores = np.asarray(scores.numpy(), dtype=np.float64)
    spec = np.asarray(spectrogram.numpy(), dtype=np.float64)
    cat_patch = patch_category_matrix(frame_scores, cat_idx)
    row_scores, row_props, category = rollup_row_metrics(frame_scores, class_names)

    fig = plt.figure(figsize=(11, 12))
    fig.suptitle(
        "Paper 3 - one segment: Welch frequency (A4) + YAMNet type (A3)\n"
        f"{wav_path.name} - first {n_samples / sr:.1f} s @ {sr} Hz | "
        f"row category = {category}",
        fontsize=11,
    )

    t_orig = np.arange(len(segment)) / sr
    ax1 = fig.add_subplot(5, 1, 1)
    ax1.plot(t_orig, segment, color="C0", linewidth=0.6)
    ax1.set_ylabel("Amplitude")
    ax1.set_xlabel("Time (s)")
    ax1.set_title("1. Waveform (original SR - Welch PSD uses this, not the 16 kHz resample)")

    ax2 = fig.add_subplot(5, 1, 2)
    if freqs.size:
        ymax = float(np.max(psd))
        for label, low, high, color in DI_BROAD_BANDS:
            ax2.axvspan(low, min(high, freqs[-1]), alpha=0.15, color=color, label=label)
        ax2.semilogy(freqs, psd + 1e-20, color="k", linewidth=0.8)
        ax2.set_xlim(AUDIO_MIN_HZ, min(AUDIO_MAX_HZ, sr / 2.0))
        ax2.set_ylabel("PSD")
        parts = [f"{label} {di_props[label]:.2f}" for label, _, _, _ in DI_BROAD_BANDS if np.isfinite(di_props[label])]
        ax2.set_title("2. Welch PSD - shaded Di bands -> freq_proportion_*  (" + ", ".join(parts) + ")")
        ax2.legend(loc="upper right", fontsize=7)
    else:
        ax2.set_title("2. Welch PSD (insufficient signal)")

    ax3 = fig.add_subplot(5, 1, 3)
    ax3.imshow(spec.T, aspect="auto", origin="lower", interpolation="nearest")
    ax3.set_ylabel("Mel band")
    ax3.set_title("3. Log-mel spectrogram (YAMNet internal - type branch only)")

    ax4 = fig.add_subplot(5, 1, 4)
    im = ax4.imshow(
        cat_patch.T,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="gray_r",
        vmin=0.0,
        vmax=max(0.5, float(np.percentile(cat_patch, 99))),
    )
    patch_padding = (PATCH_WINDOW_S / 2) / PATCH_HOP_S
    ax4.set_xlim(-patch_padding - 0.5, cat_patch.shape[0] + patch_padding - 0.5)
    ax4.set_yticks(range(len(CATEGORY_ORDER)))
    ax4.set_yticklabels(CATEGORY_ORDER, fontsize=8)
    ax4.set_xlabel("YAMNet patch index")
    ax4.set_title("4. Per-patch summed class score within each of the 7 researcher categories")
    fig.colorbar(im, ax=ax4, fraction=0.02, pad=0.02, label="Summed score")

    ax5 = fig.add_subplot(5, 1, 5)
    y_pos = np.arange(len(CATEGORY_ORDER))
    score_vals = [row_scores[c] for c in CATEGORY_ORDER]
    prop_vals = [row_props[c] for c in CATEGORY_ORDER]
    ax5.barh(y_pos - 0.2, score_vals, height=0.35, color="steelblue", label="*_score (row)")
    ax5.barh(y_pos + 0.2, prop_vals, height=0.35, color="coral", label="*_proportion (row)")
    ax5.set_yticks(y_pos)
    ax5.set_yticklabels(CATEGORY_ORDER, fontsize=8)
    ax5.set_xlim(0, max(1.0, max(score_vals + prop_vals) * 1.1))
    ax5.axvline(DETECTION_THRESHOLD, color="gray", linestyle="--", linewidth=0.8, label="0.20 threshold")
    ax5.set_xlabel("Value")
    ax5.set_title(
        f"5. Row rollup (CSV columns) - unidentify_proportion = {row_props['Unidentify']:.2f}; "
        "category from max score"
    )
    ax5.legend(loc="lower right", fontsize=7)

    output = _SndPath(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(output, dpi=150)
    print(f"Wrote {output.resolve()}")
    plt.show()
    return output


# ---------------- run ----------------
_snd_wav = _SndPath(WAV_PATH)
if _snd_wav.is_dir():
    _snd_found = sorted(_snd_wav.rglob("*.wav"))
    _snd_wav = _snd_found[0] if _snd_found else _snd_wav
if _SND_MISSING:
    print("Sound figure skipped - missing Python package(s) in this kernel: " + ", ".join(_SND_MISSING))
    print("Install them in the kernel environment (e.g. pip install soundfile tensorflow tensorflow-hub) and re-run this cell.")
elif not _snd_wav.is_file():
    print("Sound figure skipped - WAV file not found: " + str(_snd_wav))
    print("Set WAV_PATH at the top of this cell to an existing phase WAV file and re-run.")
else:
    make_sound_segment_figure(_snd_wav, SECONDS, OUTPUT)

Sound figure skipped - missing Python package(s) in this kernel: soundfile (ModuleNotFoundError), tensorflow (ModuleNotFoundError), tensorflow_hub (ModuleNotFoundError)
Install them in the kernel environment (e.g. pip install soundfile tensorflow tensorflow-hub) and re-run this cell.
